# 04 - Coach-Ready Dashboard (Streamlit)

Upgraded app: upload a clip and get **current shot + calibrated confidence + uncertainty flag + Grad-CAM why-card + rally history + predicted next shot + coaching line** - all four research questions in one screen. Uses the `model23` models.

Run notebooks 1 & 2 first (so `model23/video_model.pth` and `sequence_model.pth` exist). Run cells 1-3, then click the printed trycloudflare.com link.

In [1]:
import torch, os
assert torch.cuda.is_available(), 'No GPU active. Runtime > Change runtime type > GPU, then re-run.'
print('GPU:', torch.cuda.get_device_name(0))
from google.colab import drive
if not os.path.isdir('/content/drive/MyDrive'):
    drive.mount('/content/drive')
print('Drive ready:', os.path.isdir('/content/drive/MyDrive'))
!pip install -q streamlit opencv-python-headless
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared
print('ready')

Mounted at /content/drive
Drive ready: True
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 101.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 117.3 MB/s eta 0:00:0000:010:01
ready


In [2]:
%%writefile /content/app.py
import os, tempfile
from collections import deque
import cv2, numpy as np, torch
import torch.nn as nn, torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as transforms
import streamlit as st

CLASSES = [(0,'Short Serve','00_Short Serve'),(1,'Lift','02_Lift'),(2,'Clear','12_Clear'),(3,'Drop Shot','05_Drop Shot'),(4,'Smash','14_Smash')]
NUM_CLASSES = len(CLASSES)
ID_TO_NAME = {c[0]: c[1] for c in CLASSES}
NAME_TO_ID = {c[1]: c[0] for c in CLASSES}
PAD_TOKEN = NUM_CLASSES
SEQ_LEN = 4; NUM_FRAMES = 16; IMG = 224; CONF_THRESH = 0.60
IMAGENET_MEAN = [0.485,0.456,0.406]; IMAGENET_STD = [0.229,0.224,0.225]
MODELS = '/content/drive/MyDrive/BadmintonFYP/model23'
VIDEO_CKPT = MODELS + '/video_model.pth'
SEQUENCE_CKPT = MODELS + '/sequence_model.pth'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

WHY = {
 'Short Serve': 'low contact near the net, short forward push',
 'Lift': 'underarm lift sending the shuttle high to the rear court',
 'Clear': 'overhead/flat hit driving the shuttle to the back court',
 'Drop Shot': 'soft downward touch landing just over the net',
 'Smash': 'steep, powerful overhead downward attack',
}
COACH = {
 'Smash': 'Attack played - opponent likely forced into ',
 'Drop Shot': 'Net pressure applied - expect ',
 'Short Serve': 'Rally started - anticipate ',
 'Lift': 'Lifted to the back - prepare for ',
 'Clear': 'Cleared deep - next likely ',
}

class TemporalPoolingResNet(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        base = models.resnet18(weights=None)
        self.features = nn.Sequential(*list(base.children())[:-1])
        self.fc = nn.Linear(512, num_classes)
    def forward(self, x):
        b,t,c,h,w = x.shape
        x = x.view(b*t,c,h,w)
        x = self.features(x).view(b,t,512).mean(dim=1)
        return self.fc(x)

class ShotPredictorGRU(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, embed=32, hidden=128, layers=2, drop=0.3):
        super().__init__()
        self.emb = nn.Embedding(num_classes+1, embed, padding_idx=PAD_TOKEN)
        self.gru = nn.GRU(embed, hidden, layers, batch_first=True, dropout=drop if layers>1 else 0)
        self.fc = nn.Sequential(nn.Linear(hidden,hidden), nn.ReLU(), nn.Dropout(drop), nn.Linear(hidden,num_classes))
    def forward(self, x):
        x = self.emb(x); o,_ = self.gru(x); return self.fc(o[:,-1,:])

_norm = transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
def load_frames(path, n=NUM_FRAMES):
    cap = cv2.VideoCapture(path); total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); frames = []
    for i in np.linspace(0, max(total-1,0), n, dtype=int):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(i)); ok, fr = cap.read()
        if ok: frames.append(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB))
        elif frames: frames.append(frames[-1].copy())
    cap.release()
    while len(frames) < n: frames.append(frames[-1].copy() if frames else np.zeros((IMG,IMG,3),np.uint8))
    return frames[:n]
def letterbox(img, size=IMG):
    h,w = img.shape[:2]; s = min(size/h, size/w); nh,nw = int(h*s), int(w*s)
    r = cv2.resize(img,(nw,nh)); c = np.zeros((size,size,3),np.uint8); y,x = (size-nh)//2,(size-nw)//2; c[y:y+nh,x:x+nw] = r; return c

@st.cache_resource
def load_models():
    vm = TemporalPoolingResNet().to(DEVICE)
    vm.load_state_dict(torch.load(VIDEO_CKPT, map_location=DEVICE)); vm.eval()
    sm = None
    if os.path.exists(SEQUENCE_CKPT):
        sm = ShotPredictorGRU().to(DEVICE); sm.load_state_dict(torch.load(SEQUENCE_CKPT, map_location=DEVICE)); sm.eval()
    return vm, sm

vm, sm = load_models()
TARGET = vm.features[7]

def analyze(path):
    lb = [letterbox(f) for f in load_frames(path)]
    x = torch.stack([_norm(torch.from_numpy(f.astype(np.float32)/255.0).permute(2,0,1)) for f in lb]).unsqueeze(0).to(DEVICE)
    store = {}
    def fh(m, i, o):
        store['a'] = o
        o.register_hook(lambda g: store.update(g=g))
    h = TARGET.register_forward_hook(fh)
    vm.zero_grad()
    logits = vm(x)
    probs = torch.softmax(logits, dim=1)[0]
    order = [int(j) for j in probs.argsort(descending=True)]
    pred = order[0]
    logits[0, pred].backward()
    h.remove()
    A = store['a']; G = store['g']
    w = G.mean(dim=(2,3), keepdim=True)
    cam = torch.relu((w*A).sum(dim=1)).detach().cpu().numpy()
    t = int(cam.reshape(cam.shape[0], -1).sum(1).argmax())
    heat = cv2.resize(cam[t], (IMG, IMG)); heat = (heat-heat.min())/(heat.max()-heat.min()+1e-8)
    hc = cv2.cvtColor(cv2.applyColorMap((heat*255).astype(np.uint8), cv2.COLORMAP_JET), cv2.COLOR_BGR2RGB)
    overlay = (0.55*lb[t] + 0.45*hc).astype(np.uint8)
    top3 = [(ID_TO_NAME[i], float(probs[i])) for i in order[:3]]
    return overlay, top3

def predict_next(ids):
    h = list(ids)[-SEQ_LEN:]
    if len(h) < SEQ_LEN: h = [PAD_TOKEN]*(SEQ_LEN-len(h)) + h
    with torch.no_grad():
        p = torch.softmax(sm(torch.tensor([h], dtype=torch.long, device=DEVICE)), dim=1)[0]
    v, idx = p.topk(3)
    return [(ID_TO_NAME[int(i)], float(val)) for val, i in zip(v, idx)]

st.set_page_config(page_title='Badminton Analytics', layout='centered')
st.title('Badminton Stroke and Rally Analytics')
if 'hist' not in st.session_state: st.session_state.hist = deque(maxlen=SEQ_LEN)
hist = st.session_state.hist
with st.sidebar:
    st.header('Rally')
    st.write('History:', [ID_TO_NAME[i] for i in hist] or '-')
    if st.button('Reset rally'): hist.clear(); st.rerun()

up = st.file_uploader('Upload a shot clip', type=['mp4','avi','mov','mkv'])
if up is not None:
    suffix = os.path.splitext(up.name)[1] or '.mp4'
    tmp = tempfile.NamedTemporaryFile(delete=False, suffix=suffix); tmp.write(up.read()); tmp.close()
    st.video(tmp.name)
    overlay, top3 = analyze(tmp.name)
    os.remove(tmp.name)
    cur_label, cur_conf = top3[0]

    st.subheader('Current shot')
    if cur_conf < CONF_THRESH:
        st.warning('Uncertain: best guess ' + cur_label + (' at %.0f%% - flag for human review' % (100*cur_conf)))
    else:
        st.metric(cur_label, '%.0f%% confidence' % (100*cur_conf))

    a, b = st.columns(2)
    a.image(overlay, caption='Grad-CAM: where the model looked', use_container_width=True)
    with b:
        st.caption('Top-3')
        for lab, p in top3: st.progress(p, text=lab + (' - %.0f%%' % (100*p)))
        st.caption('Why: ' + WHY.get(cur_label, ''))

    hist.append(NAME_TO_ID[cur_label])
    st.caption('Rally history: ' + str([ID_TO_NAME[i] for i in hist]))

    if sm is not None:
        nxt = predict_next(hist)
        st.subheader('Predicted next shot')
        for lab, p in nxt: st.progress(p, text=lab + (' - %.0f%%' % (100*p)))
        st.info('Coaching: ' + COACH.get(cur_label, 'Next likely shot: ') + nxt[0][0])
    else:
        st.warning('Sequence model not found - next-shot disabled (run notebook 02).')

Writing /content/app.py


In [ ]:
import subprocess, time
subprocess.Popen(['streamlit', 'run', '/content/app.py', '--server.port', '8501', '--server.headless', 'true'])
time.sleep(8)
print('Opening public tunnel - click the trycloudflare.com URL below:')
!./cloudflared tunnel --url http://localhost:8501